# LLM Intro — Plain Text Completion & Streaming

Compare how **OpenAI**, **Anthropic**, **Google Gemini**, and **Ollama** (local) handle a simple text prompt and streaming output using each provider's native Python SDK. All four APIs do the same two jobs — send messages, get generated text back — but each SDK wraps those jobs in its own client object, call signature, and response shape, and seeing them side by side makes both the shared pattern and the differences obvious.

## Learning objectives

- Send a plain text prompt to OpenAI, Anthropic, Google Gemini, and a local Ollama model using each provider's native Python SDK.
- Locate the generated text inside each SDK's response object.
- Stream a response token by token from each provider and print it incrementally.
- Compare the four SDKs' request and response shapes side by side.

## Background

The three hosted providers authenticate with API keys read from environment variables, which must be set before launching Jupyter: `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, and `GEMINI_API_KEY`. The Ollama sections need no key but require a local Ollama server with the `llama3.2:3b-instruct-q5_K_M` model pulled (`ollama pull llama3.2:3b-instruct-q5_K_M`).

## This notebook covers

1. Plain text completion with each of the four SDKs
2. Streaming token-by-token output from each of the four SDKs
3. Review

**References:** https://platform.openai.com/docs/api-reference · https://docs.anthropic.com · https://ai.google.dev/gemini-api/docs · https://github.com/ollama/ollama-python

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math

pd.set_option('display.max_columns',100)
pd.set_option('display.max_rows',100)

plt.style.use('dark_background')

import warnings
warnings.filterwarnings('ignore')

# Shared course helpers (msds565_helpers.py lives in the repo root).
# Notebooks sit two folders below the root, so '../..' points back to it.
import sys
sys.path.append('../..')
import msds565_helpers as helpers

import os

PROMPT = 'Create a random number.'

## 1. Plain Text Completion

Every provider does the same job here — send one user message, get the completed text back. The prompt is deliberately trivial so any differences you notice are in the SDKs, not the task. Watch for three things in each section: how the client is created, what the call is named, and where the text lives in the response object.

### 1.1 OpenAI

Create an `OpenAI` client; call `client.chat.completions.create(...)` with a messages list; the text is at `resp.choices[0].message.content`.

In [ ]:
from openai import OpenAI

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
OPENAI_MODEL   = 'gpt-5'

openai_client = OpenAI(api_key=OPENAI_API_KEY)
openai_resp = openai_client.chat.completions.create(
    model=OPENAI_MODEL,
    messages=[
        {
            'role': 'user',
            'content': PROMPT
        }
    ]
)
print(openai_resp.choices[0].message.content)

684237519


### 1.2 Anthropic

Create an `Anthropic` client; call `client.messages.create(...)` — `max_tokens` is required here; the text is at `resp.content[0].text`.

In [3]:
from anthropic import Anthropic

ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')
ANTHROPIC_MODEL   = 'claude-sonnet-4-6'

anthropic_client = Anthropic(api_key=ANTHROPIC_API_KEY)
anthropic_resp = anthropic_client.messages.create(
    model=ANTHROPIC_MODEL,
    max_tokens=300,
    messages=[
        {
            'role': 'user',
            'content': PROMPT
        }
    ]
)
print(anthropic_resp.content[0].text)

Here's a random number:

**47**

(Generated arbitrarily — let me know if you need a number within a specific range or format!)


### 1.3 Google Gemini

Create a `genai.Client`; call `client.models.generate_content(...)` with `contents=` rather than a messages list; the text is at `resp.text`.

In [ ]:
from google import genai

GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
GOOGLE_MODEL   = 'gemini-2.5-flash'

client = genai.Client(api_key=GEMINI_API_KEY)
gemini_resp = client.models.generate_content(
    model=GOOGLE_MODEL,
    contents=PROMPT
)
print(gemini_resp.text)

Here's a random number:

**42**

I generated an integer between 1 and 100. Let me know if you'd like another one with different parameters (e.g., a decimal, a specific range)!


### 1.4 Ollama (local)

No client object and no API key — `ollama.chat(...)` talks to the local server and returns a plain dict; the text is at `resp['message']['content']`.

In [5]:
import ollama

OLLAMA_MODEL = 'llama3.2:3b-instruct-q5_K_M'

# Discover locally installed Ollama models:
# for m in ollama.list().models:
#     print(m.model)

ollama_resp = ollama.chat(
    model=OLLAMA_MODEL,
    messages=[
        {
            'role': 'user',
            'content': PROMPT
        }
    ]
)
print(ollama_resp['message']['content'])

The random number I generated is: **854**


## 2. Streaming — Token-by-Token Output

Instead of waiting for the full response, **streaming** lets you display tokens as they are generated — critical for responsive UIs and long outputs.

Each provider uses a slightly different streaming API, but the pattern is the same: open a stream, iterate chunks, print incrementally. The prompt below is longer than section 1's, so there are enough tokens to watch arrive.

In [13]:
PROMPT = 'Write a short poem about machine learning in exactly 12 lines.'

### 2.1 OpenAI

Set `stream=True`; iterate over chunks and read `.choices[0].delta.content`.

In [14]:
from openai import OpenAI

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
OPENAI_MODEL   = 'gpt-5'

openai_client = OpenAI(api_key=OPENAI_API_KEY)
stream = openai_client.chat.completions.create(
    model=OPENAI_MODEL,
    messages=[{'role': 'user', 'content': PROMPT}],
    stream=True
)
for chunk in stream:
    token = chunk.choices[0].delta.content
    if token:
        print(token, end='', flush=True)
print()

Data hums in grids of dawn and dusk,
Labels whisper, partial, sometimes wrong;
Gradients flow like rivers, sure and strong;
Loss is a bruise the model learns to soothe;
Weights shift, a murmuration finding truth.
Features bloom from noise, a hidden choir;
Bias lurks—a shadow stitched to wire;
We prune, apply dropout, cross-validate;
Patience backpropagates through every state;
On test-time hills, the oracle is shy,
Yet forecasts kindle sparks behind the eye;
Machine and mind co-train toward the sky.


### 2.2 Anthropic

Use `client.messages.stream()` as a context manager; iterate `.text_stream` for token strings.

In [8]:
from anthropic import Anthropic

ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')
ANTHROPIC_MODEL   = 'claude-sonnet-4-6'

anthropic_client = Anthropic(api_key=ANTHROPIC_API_KEY)
with anthropic_client.messages.stream(
    model=ANTHROPIC_MODEL,
    max_tokens=300,
    messages=[{'role': 'user', 'content': PROMPT}]
) as stream:
    for token in stream.text_stream:
        print(token, end='', flush=True)
print()

Here is a four-line poem about machine learning:

Data flows through layers deep,
Patterns learned while humans sleep,
Weights adjust with every turn,
Teaching machines the way to learn.


### 2.3 Google Gemini

Call `client.models.generate_content_stream(...)`; each chunk has a `.text` attribute.

In [9]:
from google import genai

GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
GOOGLE_MODEL   = 'gemini-2.5-flash'

client = genai.Client(api_key=GEMINI_API_KEY)
for chunk in client.models.generate_content_stream(model=GOOGLE_MODEL, contents=PROMPT):
    print(chunk.text, end='', flush=True)
print()

From vast data, patterns it discerns,
A web of logic, new knowledge earns.
Predicting futures, with careful grace,
Improving always, its silicon trace.


### 2.4 Ollama (local)

Set `stream=True`; each chunk is a dict; read `chunk['message']['content']`.

In [10]:
import ollama

OLLAMA_MODEL = 'llama3.2:3b-instruct-q5_K_M'

for chunk in ollama.chat(
    model=OLLAMA_MODEL,
    messages=[{'role': 'user', 'content': PROMPT}],
    stream=True
):
    print(chunk['message']['content'], end='', flush=True)
print()

In silicon halls, data reigns
Algorithms dance, and insights gain
Machine learning's subtle art
Weaves patterns, a digital heart


## 3. Review

All four SDKs did the same two jobs — one plain completion and one streamed completion — and the differences were entirely in the wrappers:

| | Client | Completion call | Where the text lives | Streaming pattern |
|---|---|---|---|---|
| **OpenAI** | `OpenAI(api_key=...)` | `client.chat.completions.create(...)` | `resp.choices[0].message.content` | `stream=True`; read `chunk.choices[0].delta.content` |
| **Anthropic** | `Anthropic(api_key=...)` | `client.messages.create(...)` (requires `max_tokens`) | `resp.content[0].text` | `client.messages.stream(...)` context manager; iterate `.text_stream` |
| **Google Gemini** | `genai.Client(api_key=...)` | `client.models.generate_content(...)` | `resp.text` | `client.models.generate_content_stream(...)`; read `chunk.text` |
| **Ollama (local)** | none — the library talks to the local server | `ollama.chat(model, messages)` | `resp['message']['content']` | `stream=True`; read `chunk['message']['content']` |

**Takeaways**

- **The message format is the shared core.** OpenAI, Anthropic, and Ollama all take the same list of `{'role': ..., 'content': ...}` dicts; Gemini accepts a bare string via `contents=`. Learn the pattern once and each new SDK is mostly a matter of finding the wrapper names.
- **Anthropic alone requires `max_tokens` on every call**; the others default it.
- **Ollama returns plain dicts** rather than typed response objects — and it is the only provider with no API key, because the model runs on your own machine.
- **Streaming is one idea four ways:** open an iterator, print each token with `end=''` and `flush=True`, and the response appears as it is generated.
- **Instruction following varied.** The streaming prompt asked for a poem of exactly 12 lines — only one of the four models complied. Getting *reliably structured* output takes more than asking nicely, which is exactly what the next notebook adds.

**Next:** `U3-2_LLM-2_JSONSchema.ipynb` constrains the same four providers to schema-validated JSON output.